# Dashboard v2 First-Screen Task and Feature Repair

Purpose: verify the local dashboard v2 first screen after moving Buy/Sell feature comparison and auto-update task status above the long hover inspector.

Scope: local `scripts/dashboard_server.py` workflow only. Vercel/remote deploy is intentionally out of scope.

In [1]:
from pathlib import Path
import json

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists() and (path / 'SPEC.md').exists())
V2 = ROOT / 'dashboard_data' / 'v2'
SCREENSHOT = ROOT / 'runtime' / 'production' / 'card' / 'dashboard_v2_model_features_fix_v2' / 'first_screen_feature_tasks_overflow_fixed_2026-06-14.png'
manifest = json.loads((V2 / 'manifest.json').read_text())
models = {path.stem: json.loads(path.read_text()) for path in sorted((V2 / 'models').glob('*.json'))}
summary = {
    'available_profiles': manifest['available_profiles'],
    'profile_groups': manifest.get('profile_groups'),
    'auto_update_configured': manifest.get('auto_update', {}).get('configured'),
    'auto_update_tasks': [task['name'] for task in manifest.get('auto_update', {}).get('tasks', [])],
    'auto_update_loaded': {task['name']: task.get('loaded') for task in manifest.get('auto_update', {}).get('tasks', [])},
    'safe_backtest_end': manifest.get('safe_backtest_end'),
    'latest_market_date': manifest.get('latest_market_date'),
    'screenshot_exists': SCREENSHOT.exists(),
}
summary

{'available_profiles': ['soxl_backtest_policy_reimplementation_v1',
  'state_conditioned_drawdown_rebound_reward_train_v1',
  'drawdown_rebound_reward_train_v1',
  'mode_conditioned_rebound_reward_train_v2',
  'balanced_rule_residual_v19_moe_v2'],
 'profile_groups': {'evidence_only': ['state_conditioned_drawdown_rebound_reward_train_v1',
   'drawdown_rebound_reward_train_v1',
   'mode_conditioned_rebound_reward_train_v2',
   'balanced_rule_residual_v19_moe_v2'],
  'reference': ['soxl_backtest_policy_reimplementation_v1'],
  'runnable': []},
 'auto_update_configured': True,
 'auto_update_tasks': ['preopen', 'postclose', 'dashboard_refresh'],
 'auto_update_loaded': {'preopen': True,
  'postclose': True,
  'dashboard_refresh': True},
 'safe_backtest_end': '2026-06-09',
 'latest_market_date': '2026-06-10',
 'screenshot_exists': True}

In [2]:
posttrain_hits = []
for path in V2.rglob('*.json'):
    text = path.read_text(encoding='utf-8')
    if 'post_train_review' in text or 'PostTrain' in text or 'posttrainpreview' in text.lower():
        posttrain_hits.append(str(path.relative_to(ROOT)))
assert not posttrain_hits, posttrain_hits
assert summary['auto_update_configured'] is True
assert {'preopen', 'postclose', 'dashboard_refresh'} <= set(summary['auto_update_tasks'])
assert summary['screenshot_exists'] is True
posttrain_hits

[]

In [3]:
def top_drivers(payload, n=5):
    return [row['feature'] for row in payload.get('feature_action_summary', [])[:n]]

schema_readout = {
    profile: {
        'source_kind': payload.get('source_kind'),
        'runnable': payload.get('runnable'),
        'runnable_reason': payload.get('runnable_reason'),
        'feature_count': len(payload.get('feature_columns', [])),
        'snapshot_count': len(payload.get('snapshot_columns', [])),
        'top_buy_sell_features': top_drivers(payload),
        'events': len(payload.get('events', [])),
    }
    for profile, payload in models.items()
}
assert 'panic_score' in schema_readout['soxl_backtest_policy_reimplementation_v1']['top_buy_sell_features']
assert any('relative_volume_zscore' in row['top_buy_sell_features'] for row in schema_readout.values())
assert len({tuple(row['top_buy_sell_features']) for row in schema_readout.values()}) > 1
schema_readout

{'balanced_rule_residual_v19_moe_v2': {'source_kind': 'retained_dashboard_export',
  'runnable': False,
  'runnable_reason': 'evidence_only_retained_dashboard_export',
  'feature_count': 59,
  'snapshot_count': 59,
  'top_buy_sell_features': ['relative_volume_zscore',
   'sharpe_proxy_21',
   'relative_strength_21',
   'drawdown_streak_63',
   'anchor_gap_180'],
  'events': 454},
 'drawdown_rebound_reward_train_v1': {'source_kind': 'daily_replay_evidence',
  'runnable': False,
  'runnable_reason': 'evidence_only_daily_replay',
  'feature_count': 59,
  'snapshot_count': 59,
  'top_buy_sell_features': ['relative_volume_zscore',
   'sharpe_proxy_21',
   'relative_strength_21',
   'anchor_gap_180',
   'drawdown_streak_63'],
  'events': 348},
 'mode_conditioned_rebound_reward_train_v2': {'source_kind': 'daily_replay_evidence',
  'runnable': False,
  'runnable_reason': 'evidence_only_daily_replay',
  'feature_count': 59,
  'snapshot_count': 59,
  'top_buy_sell_features': ['sharpe_proxy_21',


## Conclusion

- First-screen status now names the local auto-update tasks: `preopen`, `postclose`, and `dashboard_refresh`.
- The right stack prioritizes Buy/Sell feature comparison and Operations above the long hover Inspector.
- The Feature Driver Matrix shows model-specific driver sets: SOXLBacktest uses the qlib buy/sell baseline features such as `panic_score`, while RL evidence exports expose the 59-column feature schema led by drivers such as `relative_volume_zscore`.
- Dashboard v2 JSON contains no `post_train_review` / PostTrainPreview payload.
- Browser screenshot evidence: `runtime/production/card/dashboard_v2_model_features_fix_v2/first_screen_feature_tasks_overflow_fixed_2026-06-14.png`.